# 10 Explainability

This notebook prepares structured and text interpretation artifacts for the project report. It produces surrogate structured feature-importance tables, temporal importance summaries, note-phrase attention proxies, and clinician-facing explanation narratives.

## Explainability scope

- Structured feature importance tables
- Temporal feature importance by horizon
- Note phrase salience proxies from aggregated note windows
- Clinically interpretable narrative summaries
- Reusable CSV outputs for figures and final reporting

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/multimodal-early-sepsis') if IN_COLAB else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy

In [ ]:
import pandas as pd

from src.evaluation.explainability import (
    build_attention_phrase_table,
    build_clinical_narrative_table,
    compute_surrogate_feature_importance,
    derive_temporal_feature_importance,
)
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [ ]:
base_config_path = PROJECT_ROOT / 'configs' / 'base.yaml'
override_config_path = PROJECT_ROOT / 'configs' / 'colab.yaml' if IN_COLAB else None
config = load_config(base_config_path, override_config_path)
paths = resolve_project_paths(config)
ensure_directories(paths)
config['explainability']

## Load baseline tabular artifacts and note-window tables

In [ ]:
baseline_dir = paths['processed_data_dir'] / '06_baseline_models'
tabular_files = sorted(baseline_dir.glob('horizon_*_tabular.csv'))
assert tabular_files, 'No baseline tabular artifacts found. Run 06_baseline_models first.'

horizon_tables = {}
for horizon in config['prediction']['horizons_hours']:
    dataset_name = f'horizon_{int(horizon)}h'
    structured_path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
    if structured_path.exists():
        horizon_tables[dataset_name] = pd.read_csv(structured_path)

note_window_path = paths['processed_data_dir'] / '05_text_processing' / 'horizon_6h_note_windows.csv'
note_windows_df = pd.read_csv(note_window_path) if note_window_path.exists() else pd.DataFrame()
len(tabular_files), {k: v.shape for k, v in horizon_tables.items()}, note_windows_df.shape

## Structured feature importance

This notebook uses surrogate correlation-based importance by default so it remains lightweight and dependency-safe. It can be replaced with SHAP once the final trained model objects are serialized.

In [ ]:
tabular_df = pd.read_csv(tabular_files[0])
structured_importance_df = compute_surrogate_feature_importance(tabular_df).head(config['explainability']['top_k_features'])
structured_importance_df

## Temporal feature importance by horizon

In [ ]:
temporal_importance_df = derive_temporal_feature_importance(
    horizon_tables=horizon_tables,
    top_k=config['explainability']['top_k_features'],
)
temporal_importance_df

## Note phrase salience proxy

When true attention weights from a trained text model are not yet serialized, this phrase-frequency proxy provides a lightweight first-pass interpretation table.

In [ ]:
attention_phrase_df = build_attention_phrase_table(
    note_windows_df=note_windows_df,
    top_k_phrases=config['explainability']['top_k_phrases'],
)
attention_phrase_df

## Clinical interpretation narratives

In [ ]:
clinical_narratives_df = build_clinical_narrative_table(
    structured_importance_df,
    top_k=min(config['explainability']['top_k_features'], 10),
)
clinical_narratives_df

## Save explainability artifacts

In [ ]:
output_dir = paths['processed_data_dir'] / '10_explainability'
artifact_bundle = {
    'structured_feature_importance': structured_importance_df,
    'temporal_feature_importance': temporal_importance_df,
    'attention_phrase_proxy': attention_phrase_df,
    'clinical_narratives': clinical_narratives_df,
}
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

In [ ]:
manifest_path = paths['manifests_dir'] / '10_explainability_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='10_explainability',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'tabular_file_used': str(tabular_files[0]) if tabular_files else None,
        'note_window_available': bool(len(note_windows_df)),
    },
)
manifest_path

## Review checklist

For the final report, review:

- Which structured variables appear consistently important across horizons?
- Do the high-importance variables make clinical sense?
- Which note phrases recur in pre-sepsis windows?
- Are the explanation narratives specific enough for presentation and paper writing?
- Which artifacts should be converted into final figures?

## Project completion note

With this notebook in place, the staged Colab research pipeline is fully scaffolded from dataset setup through explainability. The remaining work is execution on the real MIMIC data, iteration on model quality, and final paper/report writing.